# Genre/IP Residual Correction

This notebook keeps the frozen post-opening baselines fixed and tests whether coarse genre/IP buckets explain residual point error or residual uncertainty. The executable logic lives in `eda/genre_ip_residual_correction.py` so the tables can be rebuilt without running the notebook UI.

In [ ]:
import os
import sys
import tempfile
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("MPLCONFIGDIR", os.path.join(tempfile.gettempdir(), "pm-box-office-matplotlib"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "eda":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "eda"))

import genre_ip_residual_correction as girc

## Rebuild Outputs

The runner exports the required diagnostics and candidate prediction tables. After-Friday daily Sat/Sun components are rescaled to the frozen AF2+C0 remaining-weekend total while preserving the baseline Sat/Sun split.

In [ ]:
outputs = girc.run_analysis(write=True)

pd.DataFrame(
    [{"table": name, "rows": len(df), "columns": len(df.columns)} for name, df in outputs.items() if isinstance(df, pd.DataFrame)]
).sort_values("table")

## Bucket Coverage

Diagnostic eligibility uses at least 10 positive examples. Model eligibility uses at least 25 positives before rolling-origin support checks are applied.

In [ ]:
coverage = outputs["coverage"]
bucket_counts = outputs["bucket_counts"]

display(coverage.sort_values(["origin", "target_name", "feature"]))
display(bucket_counts)

## Baseline Residual Summary

These summaries are descriptive only. Promotion decisions come from rolling-origin same-sample forecast error.

In [ ]:
residual_summary = outputs["residual_summary"]
display(
    residual_summary.sort_values(["origin", "target_name", "mae_residual"], ascending=[True, True, False])
)

## Point Residual Screens

Each candidate is fit with shrunk rolling bucket effects. Unsupported buckets at a forecast origin produce zero correction.

In [ ]:
point_comparison = outputs["point_comparison"]
display(
    point_comparison.sort_values(["origin", "target_name", "improvement_MAE_log_pct"], ascending=[True, True, False])
)

## Interval Scale Screens

These keep point forecasts fixed and test whether bucket effects improve empirical residual interval scale while preserving coverage.

In [ ]:
interval_comparison = outputs["interval_comparison"]
display(
    interval_comparison.sort_values(["origin", "target_name", "Winkler95_improvement_pct"], ascending=[True, True, False])
)

## Minimal Bucket Plots

Only model-eligible bucket/target pairs are plotted.

In [ ]:
targets = outputs["targets"]
eligible = coverage.loc[
    coverage["model_eligible"] & coverage["feature"].isin(girc.POINT_CANDIDATE_FEATURES)
].copy()

for (origin, target_name), feature_rows in eligible.groupby(["origin", "target_name"]):
    group = targets.loc[targets["origin"].eq(origin) & targets["target_name"].eq(target_name)].copy()
    features = list(feature_rows["feature"])
    if not features or group.empty:
        continue
    fig, axes = plt.subplots(1, len(features), figsize=(4.2 * len(features), 3.2), squeeze=False)
    for ax, feature in zip(axes.ravel(), features):
        false_values = group.loc[group[feature].fillna(False).eq(False), "target_residual"].dropna()
        true_values = group.loc[group[feature].fillna(False).eq(True), "target_residual"].dropna()
        ax.boxplot([false_values, true_values], labels=["0", "1"], showfliers=False)
        rng = np.random.default_rng(17)
        for xpos, values in [(1, false_values), (2, true_values)]:
            jitter = rng.normal(0, 0.035, size=len(values))
            ax.scatter(np.full(len(values), xpos) + jitter, values, s=12, alpha=0.45)
        ax.axhline(0, color="black", linewidth=0.8, alpha=0.45)
        ax.set_title(feature)
        ax.set_xlabel("bucket")
        ax.set_ylabel(target_name)
    fig.suptitle(f"{origin} {target_name} residuals by eligible bucket", y=1.03)
    fig.tight_layout()
    display(fig)
    plt.close(fig)

## Stability And Screening

Promotion candidates must improve the residual target, avoid OW harm where relevant, and avoid repeated major-slice degradation.

In [ ]:
screening_summary = outputs["screening_summary"]
stability_slices = outputs["stability_slices"]

display(screening_summary.sort_values(["promotion_candidate", "screen_type", "origin", "target_name"], ascending=[False, True, True, True]))
display(stability_slices.loc[stability_slices["severe_degradation"].fillna(False)].sort_values(["origin", "target_name", "candidate_model", "slice_name"]))

## Exported Files

- `data/diagnostics/genre_ip_residual_feature_coverage.csv`
- `data/diagnostics/genre_ip_residual_bucket_counts.csv`
- `data/diagnostics/genre_ip_baseline_bucket_residual_summary.csv`
- `data/diagnostics/genre_ip_residual_screening_summary.csv`
- `data/diagnostics/genre_ip_residual_point_model_comparison.csv`
- `data/diagnostics/genre_ip_residual_interval_scale_comparison.csv`
- `data/diagnostics/genre_ip_residual_stability_slices.csv`
- `data/predictions/genre_ip_residual_candidate_predictions.csv`